In [ ]:
!pip install -q chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 27.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import random
from collections import Counter
import numpy as np
import chess
import matplotlib.pyplot as plt

FIXED_FEN = "8/8/8/4k3/8/8/8/3QK3 w - - 0 1"
N_ACTIONS = 64 * 64

PLANE = {
    (chess.KING, chess.WHITE): 0,
    (chess.QUEEN, chess.WHITE): 1,
    (chess.KING, chess.BLACK): 2
}

In [ ]:
N_PLANES_BASE = 3
N_PLANES_ATTACK = 6
N_SYM = 8

def _mark(plane, squares):
    for sq in squares:
        plane[chess.square_rank(sq), chess.square_file(sq)] = 1.0

def board_to_tensor(board, attack=False):
    n = N_PLANES_ATTACK if attack else N_PLANES_BASE
    x = np.zeros((n, 8, 8), dtype=np.float32)
    for square, piece in board.piece_map().items():
        key = (piece.piece_type, piece.color)
        if key not in PLANE:
            raise ValueError(f"Quân không được hỗ trợ trong bài này: {piece}")
        x[PLANE[key], chess.square_rank(square), chess.square_file(square)] = 1.0
    if attack:
        for wq in board.pieces(chess.QUEEN, chess.WHITE):
            _mark(x[3], board.attacks(wq))
        _mark(x[4], board.attacks(board.king(chess.WHITE)))
        bk = board.king(chess.BLACK)
        b = board.copy(stack=False)
        b.turn = chess.BLACK
        for m in b.legal_moves:
            if m.from_square == bk:
                _mark(x[5], [m.to_square])
    return x

def _sym_square(sq, k):
    r, f = chess.square_rank(sq), chess.square_file(sq)
    if k & 4: r, f = f, r
    if k & 1: f = 7 - f
    if k & 2: r = 7 - r
    return r * 8 + f

SYM_PERM = np.array([[_sym_square(sq, k) for sq in range(64)] for k in range(N_SYM)], dtype=np.int64)
assert all(sorted(p.tolist()) == list(range(64)) for p in SYM_PERM)
SYM_OBS_INV = np.argsort(SYM_PERM, axis=1)
_ACT = np.arange(N_ACTIONS)
SYM_ACT_PERM = SYM_PERM[:, _ACT // 64] * 64 + SYM_PERM[:, _ACT % 64]
SYM_ACT_INV = np.argsort(SYM_ACT_PERM, axis=1)

def augment_obs(x, k):
    """x: (B, P, 8, 8), k: (B,) — biến đổi từng mẫu theo phép đối xứng k."""
    B, P = x.shape[0], x.shape[1]
    idx = np.broadcast_to(SYM_OBS_INV[k][:, None, :], (B, P, 64))
    return np.take_along_axis(x.reshape(B, P, 64), idx, axis=2).reshape(B, P, 8, 8)

def augment_mask(m, k):
    """m: (B, 4096) bool, k: (B,)."""
    return np.take_along_axis(m, SYM_ACT_INV[k], axis=1)

def augment_action(a, k):
    """a: (B,) chỉ số action, k: (B,)."""
    return SYM_ACT_PERM[k, a]

In [ ]:
def move_to_index(move):
  return move.from_square * 64 + move.to_square

def index_to_move(index):
  return chess.Move(index // 64, index % 64)

def legal_mask(board):
  mask = np.zeros(N_ACTIONS, dtype=bool)
  for m in board.legal_moves:
    mask[move_to_index(m)] = True
  return mask
b = chess.Board(FIXED_FEN)
m = chess.Move.from_uci("e1e5")
print(move_to_index(m), index_to_move(move_to_index(m)), legal_mask(b).sum())

292 e1e5 21


In [ ]:
def sample_position(rng, level=2, max_tries=10000):
  if level == 0:
    return chess.Board(FIXED_FEN)

  for _ in range(max_tries):
    white_king, white_queen, black_king = rng.sample(range(64),3)
    if level == 1:
      r, f = chess.square_rank(black_king), chess.square_file(black_king)
      if r not in (0, 7) and f not in (0, 7):
        continue
    board = chess.Board(None)
    board.set_piece_at(white_king, chess.Piece(chess.KING, chess.WHITE))
    board.set_piece_at(black_king, chess.Piece(chess.KING, chess.BLACK))
    board.set_piece_at(white_queen, chess.Piece(chess.QUEEN, chess.WHITE))
    board.turn = chess.WHITE

    if not board.is_valid() or board.is_game_over():
      continue
    return board

  raise RuntimeError("Không sinh được vị trí hợp lệ")
rng = random.Random(0)
print(sample_position(rng, level=2))


. . . . . . Q .
. . . k . . . .
. . . . . . . .
. K . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .


In [ ]:
def _center_distance(sq):
    """Khoảng cách Chebyshev từ sq tới tâm bàn cờ (cao = gần biên)."""
    r, f = chess.square_rank(sq), chess.square_file(sq)
    return max(abs(r - 3.5), abs(f - 3.5))

def _chebyshev(sq1, sq2):
    r1, f1 = chess.square_rank(sq1), chess.square_file(sq1)
    r2, f2 = chess.square_rank(sq2), chess.square_file(sq2)
    return max(abs(r1 - r2), abs(f1 - f2))

def _potential(board):
    """
    Hàm potential Φ(s) ∈ [0, 1].
    Giá trị cao = thế trận tốt cho trắng.
    """
    bk = board.king(chess.BLACK)
    wk = board.king(chess.WHITE)
    wq_squares = list(board.pieces(chess.QUEEN, chess.WHITE))
    if not wq_squares:
        return 0.0
    wq = wq_squares[0]

    bk_edge = _center_distance(bk) / 3.5

    king_close = 1.0 - _chebyshev(wk, bk) / 7.0

    queen_king = 1.0 - _chebyshev(wq, bk) / 7.0

    return 0.5 * bk_edge + 0.3 * king_close + 0.2 * queen_king

b_test = chess.Board(FIXED_FEN)
print(f"Potential của vị trí ban đầu: {_potential(b_test):.3f}")

Potential của vị trí ban đầu: 0.286


In [ ]:
def defender_move(board, rng):
    scored = []
    for m in list(board.legal_moves):
        board.push(m)
        if board.is_stalemate() or board.is_insufficient_material():
            s = 500.0
        else:
            allows_mate = False
            for wm in list(board.legal_moves):
                board.push(wm)
                allows_mate = board.is_checkmate()
                board.pop()
                if allows_mate:
                    break
            s = -500.0 if allows_mate else -_center_distance(board.king(chess.BLACK))
        board.pop()
        scored.append((s, m))
    best = max(s for s, _ in scored)
    return rng.choice([m for s, m in scored if s == best])

In [ ]:
class KQKEnv:
    """
    Trắng: vua + hậu. Đen: vua, đi ngẫu nhiên trong các nước hợp lệ.
    Reward: chiếu hết +win | mất hậu / stalemate / hết giờ / kết cục khác: −1 | mỗi nước step_penalty
    Shaping (Ng et al. 1999): F = w·(γ·Φ(s') − Φ(s)), Φ(kết thúc) = 0.
    """
    n_actions = N_ACTIONS

    def __init__(self, max_moves=50, win_reward=5.0, lose_queen_reward=-1.0,
                 stalemate_reward=-1.0, timeout_reward=-1.0, step_penalty=-0.01,
                 level=0, seed=None, shaping_weight=0.3, shaping_gamma=0.95, attack=False, opponent="random"):
        self.max_moves = max_moves
        self.win_reward = win_reward
        self.lose_queen_reward = lose_queen_reward
        self.stalemate_reward = stalemate_reward
        self.timeout_reward = timeout_reward
        self.step_penalty = step_penalty
        self.level = level
        self.rng = random.Random(seed)
        self.shaping_weight = shaping_weight
        self.shaping_gamma = shaping_gamma
        self.attack = attack
        self.n_planes = N_PLANES_ATTACK if attack else N_PLANES_BASE
        self.observation_shape = (self.n_planes, 8, 8)
        self.board = None
        self.moves = 0
        self._prev_potential = 0.0
        self.opponent = opponent

    def _obs(self):
        return board_to_tensor(self.board, self.attack)

    def _info(self, result):
        return {"legal_mask": legal_mask(self.board), "result": result}

    def _shape(self, base_reward, phi_next):
        if self.shaping_weight == 0:
            return base_reward
        F = self.shaping_weight * (self.shaping_gamma * phi_next - self._prev_potential)
        self._prev_potential = phi_next
        return base_reward + F

    def _finish(self):
        name = self.board.outcome().termination.name.lower()
        if name == "checkmate":
            reward = self.win_reward
        elif name == "insufficient_material":
            reward = self.lose_queen_reward
        elif name == "stalemate":
            reward = self.stalemate_reward
        else:
            reward = self.timeout_reward
        reward = self._shape(reward, phi_next=0.0)
        return self._obs(), reward, True, False, self._info(name)

    def reset(self, seed=None, fen=None, level=None, opponent=None):
        if seed is not None:
            self.rng = random.Random(seed)
        if opponent is not None:
            self.opponent = opponent
        if fen is not None:
            self.board = chess.Board(fen)
        else:
            lvl = self.level if level is None else level
            self.board = sample_position(self.rng, lvl)
        self.moves = 0
        self._prev_potential = _potential(self.board)
        return self._obs(), self._info(None)

    def step(self, action):
        move = index_to_move(int(action))
        if move not in self.board.legal_moves:
            raise ValueError(f"Nước không hợp lệ: {move.uci()}. Hãy che (mask) trước khi chọn action.")
        self.board.push(move)
        self.moves += 1
        if self.board.is_game_over():
            return self._finish()
        if self.opponent == "defender":
            self.board.push(defender_move(self.board, self.rng))
        else:
            self.board.push(self.rng.choice(list(self.board.legal_moves)))
        if self.board.is_game_over():
            return self._finish()
        if self.moves >= self.max_moves:
            reward = self._shape(self.timeout_reward, phi_next=0.0)
            return self._obs(), reward, False, True, self._info("max_moves")
        reward = self._shape(self.step_penalty, phi_next=_potential(self.board))
        return self._obs(), reward, False, False, self._info(None)

    def render(self):
        print(self.board)

In [ ]:
def random_policy(env, rng):
  moves = list(env.board.legal_moves)
  return move_to_index(rng.choice(moves))

def greedy_policy(env, rng):
  board = env.board
  moves = list(board.legal_moves)
  for mv in moves:
    board.push(mv)
    mate = board.is_checkmate()
    board.pop()
    if mate:
      return move_to_index(mv)
  return move_to_index(rng.choice(moves))


def play_games(env, policy, n_games = 5000, level = 0, seed = 0):
  rng = random.Random(seed)
  results = Counter()
  lengths, rewards = [], []
  for g in range(n_games):
      env.reset(seed=seed + g, level = level)
      done, n_steps, total = False, 0, 0.0
      while not done:
        action = policy(env, rng)
        _, r, terminated, truncated, info = env.step(action)
        total += r
        n_steps += 1
        done = terminated or truncated
      results[info["result"]] += 1
      lengths.append(n_steps)
      rewards.append(total)
  return {
      "policy": policy.__name__, "level": level, "n_games": n_games,
      "results": dict(results),
      "checkmate_rate": results["checkmate"] / n_games,
      "queen_loss_rate": results["insufficient_material"] / n_games,
      "stalemate_rate": results["stalemate"] / n_games,
      "timeout_rate": results["max_moves"] / n_games,
      "mean_length": float(np.mean(lengths)),
      "mean_reward": float(np.mean(rewards)),
      "other_rate": sum(v for k, v in results.items()
                          if k not in ("checkmate", "insufficient_material",
                                       "stalemate", "max_moves")) / n_games,
  }

In [ ]:
def run_tests():
    b = chess.Board(FIXED_FEN)
    x = board_to_tensor(b)
    assert x.shape == (3, 8, 8)
    assert x.sum(axis=(1, 2)).tolist() == [1.0, 1.0, 1.0]
    assert x[1, 0, 3] == 1.0 and x[0, 0, 4] == 1.0 and x[2, 4, 4] == 1.0
    xa = board_to_tensor(b, attack=True)
    assert xa.shape == (6, 8, 8) and np.array_equal(xa[:3], x)
    for mv in b.legal_moves:
        assert index_to_move(move_to_index(mv)) == mv
    assert legal_mask(b).sum() == b.legal_moves.count()

    rng = random.Random(7)
    for _ in range(50):
        bd = sample_position(rng, 2)
        obs = board_to_tensor(bd, attack=True)
        mask = legal_mask(bd)
        for k in range(N_SYM):
            tb = chess.Board(None)
            for sq, pc in bd.piece_map().items():
                tb.set_piece_at(int(SYM_PERM[k, sq]), pc)
            tb.turn = bd.turn
            assert np.array_equal(board_to_tensor(tb, attack=True),
                                  augment_obs(obs[None], np.array([k]))[0]), k
            tmask = legal_mask(tb)
            assert np.array_equal(tmask, augment_mask(mask[None], np.array([k]))[0]), k
            for a in np.flatnonzero(mask)[:3]:
                assert tmask[int(SYM_ACT_PERM[k, a])]

    env = KQKEnv(shaping_weight=0.0)
    env.reset(fen="7k/8/5K2/8/8/8/8/6Q1 w - - 0 1")
    _, r, term, trunc, info = env.step(move_to_index(chess.Move.from_uci("g1g7")))
    assert term and not trunc and info["result"] == "checkmate" and r == env.win_reward, (r, info["result"])

    env.reset(fen="k7/2K5/7Q/8/8/8/8/8 w - - 0 1")
    _, r, term, trunc, info = env.step(move_to_index(chess.Move.from_uci("h6b6")))
    assert term and info["result"] == "stalemate" and r == env.stalemate_reward, (r, info["result"])

    env.reset()
    try:
        env.step(move_to_index(chess.Move.from_uci("a1a2")))
        raise AssertionError("Lẽ ra phải báo lỗi nước không hợp lệ")
    except ValueError:
        pass

    env2 = KQKEnv(max_moves=1, shaping_weight=0.0)
    env2.reset(fen=FIXED_FEN)
    _, r, term, trunc, info = env2.step(move_to_index(chess.Move.from_uci("e1e2")))
    assert trunc and not term and info["result"] == "max_moves" and r == env2.timeout_reward, (r, info["result"])

    env3 = KQKEnv(max_moves=50, shaping_weight=0.0)
    expected = {"checkmate": env3.win_reward, "insufficient_material": env3.lose_queen_reward,
                "stalemate": env3.stalemate_reward, "max_moves": env3.timeout_reward}
    n_qloss = 0
    for g in range(300):
        env3.reset(seed=g, level=0)
        prng = random.Random(g)
        done = False
        while not done:
            _, r, term, trunc, info = env3.step(random_policy(env3, prng))
            done = term or trunc
        assert r == expected.get(info["result"], env3.timeout_reward), (info["result"], r)
        n_qloss += info["result"] == "insufficient_material"
    assert n_qloss > 0, "Lẽ ra phải có ván mất hậu khi chơi ngẫu nhiên"

    gamma, w = 0.95, 0.3
    for g in range(50):
        e0 = KQKEnv(shaping_weight=0.0, shaping_gamma=gamma, level=2)
        e1 = KQKEnv(shaping_weight=w, shaping_gamma=gamma, level=2)
        e0.reset(seed=g)
        e1.reset(seed=g)
        phi0 = _potential(e1.board)
        prng = random.Random(g)
        disc, t, done = 0.0, 0, False
        while not done:
            a = random_policy(e0, prng)
            _, r0, term0, trunc0, _ = e0.step(a)
            _, r1, term1, trunc1, _ = e1.step(a)
            assert (term0, trunc0) == (term1, trunc1)
            disc += gamma ** t * (r1 - r0)
            t += 1
            done = term0 or trunc0
        assert abs(disc + w * phi0) < 1e-9, (disc, -w * phi0)

    print("Tất cả kiểm tra đều qua.")

if __name__ == "__main__":
    run_tests()

Tất cả kiểm tra đều qua.
